# Module 2 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Canvas where you downloaded this file.

*Provide the output **exactly** as requested*

In [1]:
from pprint import pprint

## Local Search - Genetic Algorithm

There are some key ideas in the Genetic Algorithm.

First, there is a problem of some kind that either *is* an optimization problem or the solution can be expressed in terms of an optimization problem.
For example, if we wanted to minimize the function

$$f(x) = \sum (x_i - 0.5)^2$$

where $n = 10$.
This *is* an optimization problem. Normally, optimization problems are much, much harder.

![Eggholder](http://www.sfu.ca/~ssurjano/egg.png)!

The function we wish to optimize is often called the **objective function**.
The objective function is closely related to the **fitness** function in the GA.
If we have a **maximization** problem, then we can use the objective function directly as a fitness function.
If we have a **minimization** problem, then we need to convert the objective function into a suitable fitness function, since fitness functions must always mean "more is better".

Second, we need to *encode* candidate solutions using an "alphabet" analogous to G, A, T, C in DNA.
This encoding can be quite abstract.
You saw this in the Self Check.
There a floating point number was encoded as bits, just as in a computer and a sophisticated decoding scheme was then required.

Sometimes, the encoding need not be very complicated at all.
For example, in the real-valued GA, discussed in the Lectures, we could represent 2.73 as....2.73.
This is similarly true for a string matching problem.
We *could* encode "a" as "a", 97, or '01100001'.
And then "hello" would be:

```
["h", "e", "l", "l", "o"]
```

or

```
[104, 101, 108, 108, 111]
```

or

```
0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1
```

In Genetics terminology, this is the **chromosome** of the individual. And if this individual had the **phenotype** "h" for the first character then they would have the **genotype** for "h" (either as "h", 104, or 01101000).

To keep it straight, think **geno**type is **genes** and **pheno**type is **phenomenon**, the actual thing that the genes express.
So while we might encode a number as 10110110 (genotype), the number itself, 182, is what goes into the fitness function.
The environment operates on zebras, not the genes for stripes.

## String Matching

You are going to write a Genetic Algorithm that will solve the problem of matching a target string (at least at the start).
Now, this is kind of silly because in order for this to work, you need to know the target string and if you know the target string, why are you trying to do it?
Well, the problem is *pedagogical*.
It's a fun way of visualizing the GA at work, because as the GA finds better and better candidates, they make more and more sense.

Now, string matching is not *directly* an optimization problem so this falls under the general category of "if we convert the problem into an optimization problem we can solve it with an optimization algorithm" approach to problem solving.
This happens all the time.
We have a problem.
We can't solve it.
We convert it to a problem we *can* solve.
In this case, we're using the GA to solve the optimization part.

And all we need is some sort of measure of the difference between two strings.
The only constraint for our objective function is that it must calculate the score based on element to element (character to character) comparisons with no global transformations of the candidate or target strings.
That measure becomes our **objective function** and we can use it with the Genetic Algorithm.

Since it is probably easier to come up with a score that measures how far apart the two strings are, we will end up with an objective function that represents a **minimization problem**.
Because a fitness function must always be "more is better", we'll need to convert our objective function to a proper fitness function as well.

And since this is a GA, we need a **genotype**.
The genotype for this problem is a list of "characters" (individual letters aren't special in Python like they are in some other languages):

```
["h", "e", "l", "l", "o"]
```

and the **phenotype** is the resulting string:

```
"hello"
```

In addition to the generic code and problem specific loss function, you'll need to pick parameters for the run.
These parameters include:

1. population size
2. number of generations
3. probability of crossover
4. probability of mutation

You will also need to pick a selection algorithm, either roulette wheel or tournament selection.
In the later case, you will need a tournament size.
This is all part of the problem.

Every **ten** (10) generations, you should print out the fitness, genotype, and phenotype of the best individual in the population for the specific generation.
The function should return the best individual *of the entire run*, using the same format.

In [2]:
ALPHABET = "abcdefghijklmnopqrstuvwxyz "

In [3]:
#import necessary packages
import random
from operator import itemgetter
from typing import List, Tuple, Dict, Callable
import itertools

<a id="fitness_evaluation_normal"></a>
## fitness_evaluation_normal

In order to check how close a given string is to the target string, a fitness score must be calculated. In this function, the fitness score is calculated as the number of correct characters in the correct position in the child chromosome, compared to the target string. 

* **target** String: target string to be discovered by algorithm.
* **child** List[str]: child string to be evaluated for fitness
* **alphabet** String: string of all possible characters to be used by algorithm.
  
**returns** **fitness_score** int: fitness score evaluated for the inputted child string

In [4]:
def fitness_evaluation_normal(target: str, child: list[str], alphabet: str):

    fitness_score = 0
    
    for target_i, child_i in zip(target, child):
        if target_i == child_i:
            fitness_score = fitness_score + 1

    return fitness_score

In [5]:
test_target = 'this is a test'

test_child1 = 'this is a test'
assert fitness_evaluation_normal(test_target, test_child1, ALPHABET) == 14

test_child2 = 'this is a rest'
assert fitness_evaluation_normal(test_target, test_child2, ALPHABET) == 13

test_child3 = '12345678987654'
assert fitness_evaluation_normal(test_target, test_child3, ALPHABET) == 0


<a id="fitness_evaluation_reverse"></a>
## fitness_evaluation_reverse

In order to check how close a given string is to the target string, a fitness score must be calculated. In this function, the fitness score is as the number of correct characters in the reverse position in the child chromosome, compared to the target string. 

* **target** String: target string to be discovered by algorithm.
* **child** List[str]: child string to be evaluated for fitness 
* **alphabet** String: string of all possible characters to be used by algorithm.
  
**returns** **fitness_score** int: fitness score evaluated for the inputted child string

In [6]:
def fitness_evaluation_reverse(target: str, child: list[str], alphabet: str):

    fitness_score = 0

    for i in range(0, len(child)):
        j = len(target) - 1 - i  #index to read target string in reverse order
        if child[i] == target[j]:
            fitness_score = fitness_score + 1

    return fitness_score

In [7]:
test_target = 'this is a test'

test_child1 = 'tset a si siht'
assert fitness_evaluation_reverse(test_target, test_child1, ALPHABET) == 14

test_child2 = 'tset---si a siht'
assert fitness_evaluation_reverse(test_target, test_child2, ALPHABET) == 7

test_child3 = '12345678987654'
assert fitness_evaluation_reverse(test_target, test_child3, ALPHABET) == 0


<a id="fitness_evaluation_roth13"></a>
## fitness_evaluation_rot13

In order to check how close a given string is to the target string, a fitness score must be calculated. ROT13 is an encryption function that shifts the alphabet by 13 characters. In this function, the fitness score checks how close a given string is to the ROT13 encryption of the target string. 

* **target** String: target string to be discovered by algorithm.
* **child** String: child string to be evaluated for fitness
* **alphabet** String: string of all possible characters to be used by algorithm.
  
**returns** **fitness_score** int: fitness score evaluated for the inputted child string

In [8]:
def fitness_evaluation_rot13(target: str, child: list[str], alphabet: str):

    fitness_score = 0

    for i in range(0, len(child)):
        index = alphabet.index(target[i])
        j = (index + 13) % 26
        if child[i] == alphabet[j]:
            fitness_score = fitness_score + 1

    return fitness_score

In [9]:
test_target = 'thisisatest'
test_alphabet = 'abcdefghijklmnopqrstuvwxyz'

test_child1 = 'guvfvfngrfg'
assert fitness_evaluation_rot13(test_target, test_child1, test_alphabet) == 11

test_child2 = 'guoflmngrwg'
assert fitness_evaluation_rot13(test_target, test_child2, test_alphabet) == 7

test_child3 = '12345678987'
assert fitness_evaluation_rot13(test_target, test_child3, test_alphabet) == 0

<a id="create_population"></a>
## create_population

In the genetic algorithm, one starts with a random population that will be used to crossover and produce the next generation. The population size (n) is determined before running the genetic algorithm. The population consists of n number of chromosomes. Each chromosome is the same length as the target string. Each chromosome is a random assortment of the potential characters provided by the alphabet. 

* **parameters** Dict[str, float]: dictionary of parameters required for algorithm
* **target** String: target string to be discovered by algorithm
* **alphabet** String: string of all possible characters to be used by algorithm
* **fitness_evaluation** Callable: chosen fitness evaluation function
  
**returns** random **population**: List[List[List[str]], int]

In [10]:
def create_population(parameters: dict[str, float], target: str, alphabet: str, fitness_evaluation: Callable):
    target_length = len(target)
    population = []
    population_size = parameters['population_size']

    #create n chromosomes to populate random populations
    for i in range(parameters['population_size']):
        chromosome = []
        for j in range(target_length):
            chromosome.append(random.choice(alphabet))
        fitness = fitness_evaluation(target, chromosome, alphabet)
        population.append([chromosome, fitness])
        
    return population

In [11]:
test_parameters = {'population_size': 10, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}
test_target1 = 'this is a test string'
test_population = create_population(test_parameters, test_target1, ALPHABET, fitness_evaluation_normal)

assert len(test_target1) == len(test_population[0][0])
assert len(test_population) == test_parameters['population_size']

test_target2 = ''
test_population = create_population(test_parameters, test_target2, ALPHABET, fitness_evaluation_normal)
assert len(test_population[0][0]) == 0

<a id="roulette_selection"></a>
## roulette_selection

In the genetic algorithm, it is important to select for the fitter population members to become parents and crossover into the next generation. The roulette wheel selection is one of the many methods of selecting the fitter population for propagation. The roulette wheel method is a stochastic method where the probability of selecting a chromosome into the fit population is proportional to the fitness of the chromosome in relation to the remaining population.

* **population** List[List[List[str]], int]: population to be evaluated
* **target** String: target string to be discovered by algorithm
* **fitness_evaluation** Callable: chosen fitness evaluation function
* **alphabet** String: string of all possible characters to be used by algorithm
  
**returns** new fit **population** List[List[List[str]], int]: population selected by roulette

In [12]:
def roulette_selection(population: list[list[str, int]], target: str, fitness_evaluation: Callable, alphabet: str):
    chromosomes = []
    fitness_score = []
    fit_population = []

    #split population into chromosomes and fitness score
    for x in population:
        chromosomes.append(x[0])
        fitness_score.append(x[1])

    #calculate the individual fitness proportional to total fitness, and associated probability
    total_fitness = float(sum(fitness_score))
    relative_fitness = [score/total_fitness for score in fitness_score]
    probability = [sum(relative_fitness[:i + 1]) for i in range(len(relative_fitness))]

    #select for N/2 fit chromosomes 
    for i in range(int(len(chromosomes)/2)):
        random_choice = random.uniform(0, 1)
        for (i, chromosome) in enumerate(population):
            if random_choice <= probability[i]:
                fit_score = fitness_evaluation(target, chromosome[0], alphabet)
                fit_population.append([chromosome[0], fit_score])
                break
                
    return fit_population

In [13]:
test_target = 'abcde'
test_population = [
    [['a', 'b', 'c', 'd', 'e'], 5],
    [['d', 'b', 'x', 'd', 'm'], 3],
    [['q', 'q', 'm', 'g', 'r'], 0],
    [['y', 'r', 'e', 'b', 'l'], 0],
    [['a', 'b', 'c', 'd', 'm'], 4],
    [['u', 'y', 'c', 'e', 'k'], 1]
    ]
new_test_population = roulette_selection(test_population, test_target, fitness_evaluation_normal, ALPHABET)
assert len(new_test_population) == 3
assert len(new_test_population[0][0]) == 5
assert isinstance(new_test_population[0][1], int)


<a id="mutate"></a>
## mutate

In genetic algorithms, just as in biology and evolutionary practices, the recombined children can undergo random mutation to introduce genetic diversity into the population. If a randomly generated value is less than the indicated mutation rate, a random character is chosen to mutate the chromsome at a random location. 

* **parameters** Dict[str, float]: dictionary of parameters required for algorithm
* **alphabet** String: string of all possible characters to be used by algorithm
* **child** List[str]: child chromosome to be mutated
  
**returns** mutated or unmutated **child**: List[str] 

In [14]:
def mutate(parameters: dict[str, float], alphabet: str, child: list[str]):
    
    if random.uniform(0, 1) <= parameters['mutation_rate']:
        mutation_index = random.randint(0, len(child) - 1)
        new_character = random.choice(alphabet)
        child[mutation_index] = new_character
    
    return child

In [15]:
test_parameters = {'population_size': 10, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}
test_child = ['t','h','i','s',' ','i','s',' ','a',' ','t','e','s','t']
test_child2 = mutate(test_parameters, ALPHABET, test_child)
assert len(test_child) == len(test_child2)

matching = 0
for test_child_i, test_child2_i in zip(test_child, test_child2):
    if test_child_i == test_child2_i:
        matching = matching + 1
    else: 
        assert test_child2_i in ALPHABET
        
assert (matching - len(test_child)) <= 1


<a id="crossover"></a>
## crossover

As in biology and evolution, the genetic algoorithm undergoes crossover, or recombination. It is a method of combining genetic information from two parent chromosomes to create two children chromosomes. This is done by randomly generating a crossover index around which the two parents swap genetic material. This function uses a one-point crossover to create children chromsomes. The function also utilizes a crossover rate to dictate if a parent chromosome undergoes crossover. If crossover does not occur, the parents themselves join the new generation. The children can also undergo mutation to introduce new characters into the offspring, at a random location. 

* **parameters** Dict[str, float]: dictionary of parameters required for algorithm
* **target** String: target string to be discovered by algorithm.
* **alphabet** String: string of all possible characters to be used by algorithm
* **parent_1** List[List[str], int]: first parent to be crossed over
* **parent_2** List[List[str], int]: second parent to be crossed over
* **fitness_evaluation** Callable: chosen fitness evaluation function
  
**returns** initial **population**: List[List[str, int]] 

In [16]:
def crossover(parameters: dict[str, float], target: str, alphabet: str, parent_1: list[list[str], int], parent_2: list[list[str], int], fitness_evaluation: Callable):
    crossover_index = random.randint(0, len(parent_1[0]) - 1)

    if random.uniform(0, 1) <= parameters['crossover_rate']:
        child_1 = parent_1[0][:crossover_index]
        child_1.extend(parent_2[0][crossover_index:])
    
        child_2 = parent_2[0][:crossover_index]
        child_2.extend(parent_1[0][crossover_index:])
    else:
        child_1 = parent_1[0]
        child_2 = parent_2[0]
        
    child_1 = mutate(parameters, alphabet, child_1)
    child_2 = mutate(parameters, alphabet, child_2)

    child1 = [child_1, fitness_evaluation(target, child_1, alphabet)] 
    child2 = [child_2, fitness_evaluation(target, child_2, alphabet)] 
    
    return child1, child2

In [17]:
test_parameters = {'population_size': 10, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}
test_target1 = 'this is a test string'
test_parent_1 = [['w', 'e', 'j', 'j', 'w', 'o', 'c', 'g', 't', 'm', 'q', ' ', 'i', 'z', 'y', 'r', 'n', 'd', 'o'],0]
test_parent_2 = [[' ', ' ', 'i', 'r', 'b', 'x', 'o', 'q', 'm', 't', 'b', 's', 'k', 'i', ' ', 'p', 'o', 'i', 'u'],0]

test_child_1, test_child_2 = crossover(test_parameters, test_target1, ALPHABET, test_parent_1, test_parent_2, fitness_evaluation_normal)
assert len(test_parent_1[0]) == len(test_child_1[0])
assert len(test_parent_2[0]) == len(test_child_2[0])
assert isinstance(test_child_1[1], int)

<a id="genetic_algorithm"></a>
### genetic_algorithm

The genetic algorithm, inspired by natural selection and evolution, is often used to solve search problems and optimization problems. The algorithm uses functions such as crossover, mutation and selection to navigate the fittest results until a target goal is reached. This function crossovers and mutates a string until the target string is met, using a fitness evaluator to gage the progress towards the end goal. 

* **parameters** Dict[str, float]: dictionary of parameters required for algorithm
* **target** String: target string to be discovered by algorithm.
* **alphabet** String: string of all possible characters to be used by algorithm
* **fitness_evaluation** Callable: chosen fitness evaluation function

**returns** **best chromosome**: Dict[str, str] or None

In [18]:
def genetic_algorithm(parameters: dict[str, float], target: str, alphabet: str, fitness_evaluation: Callable): # add your formal parameters

    population = create_population(parameters, target, alphabet, fitness_evaluation)
    
    for i in range(parameters['generation_count']):
        fittest_population = roulette_selection(population, target, fitness_evaluation, alphabet)
        new_generation = []
        
        for j in range(int(len(population)/2)):
            parent_1 = random.choice(fittest_population)
            parent_2 = random.choice(fittest_population)
            child_1, child_2 = crossover(parameters, target, alphabet, parent_1, parent_2, fitness_evaluation)
            new_generation.append(child_1)
            new_generation.append(child_2)

        new_generation = sorted(new_generation, key = itemgetter(1), reverse=True)

        if new_generation[0][1] == len(target):
            return {'Generation': i, 'Best Fitness Score': str(new_generation[0][1]), 'Best Genotype': str(new_generation[0][0]), 'Best Phenotype': ''.join(str(x) for x in new_generation[0][0])}
        
        population = new_generation

        if i % 10 == 0:
            print("Generation: " + str(i))
            print("Best Fitness Score: " + str(new_generation[0][1]))
            print("Best Genotype: " + str(new_generation[0][0]))
            print("Best Phenotype: " + ''.join(str(x) for x in new_generation[0][0]))
                  
            
    return None 

## Problem 1

The target is the string "this is so much fun".
The challenge, aside from implementing the basic algorithm, is deriving a fitness function based on "b" - "p" (for example).
The fitness function should come up with a fitness score based on element to element comparisons between target v. phenotype.

In [19]:
target1 = "this is so much fun"

In [20]:
parameters1 = {'population_size': 500, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}

In [21]:
result1 = genetic_algorithm(parameters1, target1, ALPHABET, fitness_evaluation_normal)

Generation: 0
Best Fitness Score: 4
Best Genotype: ['d', 'x', 'l', 'd', ' ', 'b', 's', 'x', 's', 'o', 'x', 'f', 'l', 'v', 'd', 'l', 'c', ' ', 'c']
Best Phenotype: dxld bsxsoxflvdlc c
Generation: 10
Best Fitness Score: 11
Best Genotype: ['t', 'h', 'i', 'x', 'q', 'i', 'm', ' ', 's', 'o', 'k', 'b', 'u', 'c', 'h', 'p', 'p', 'u', 'k']
Best Phenotype: thixqim sokbuchppuk
Generation: 20
Best Fitness Score: 15
Best Genotype: ['t', 'h', ' ', 's', ' ', 'i', 'm', ' ', 's', 'o', ' ', 'm', 'u', 'c', 'h', ' ', 'v', 'k', 'n']
Best Phenotype: th s im so much vkn
Generation: 30
Best Fitness Score: 15
Best Genotype: ['t', 'h', 'b', 'e', ' ', 'i', 'm', ' ', 's', 'o', ' ', 'm', 'u', 'c', 'h', ' ', 'f', 'v', 'n']
Best Phenotype: thbe im so much fvn
Generation: 40
Best Fitness Score: 16
Best Genotype: ['t', 'h', ' ', 's', ' ', 'i', 's', 'g', 's', 'o', ' ', 'm', 'u', 'c', 'h', ' ', 'f', 'k', 'n']
Best Phenotype: th s isgso much fkn
Generation: 50
Best Fitness Score: 16
Best Genotype: ['t', 'h', ' ', 's', ' '

In [22]:
pprint(result1, compact=True)

{'Best Fitness Score': '19',
 'Best Genotype': "['t', 'h', 'i', 's', ' ', 'i', 's', ' ', 's', 'o', ' ', "
                  "'m', 'u', 'c', 'h', ' ', 'f', 'u', 'n']",
 'Best Phenotype': 'this is so much fun',
 'Generation': 153}


## Problem 2

You should have working code now.
The goal here is to think a bit more about fitness functions.
The target string is now, 'nuf hcum os si siht'.
This is obviously target #1 but reversed.
If we just wanted to match the string, this would be trivial.
Instead, this problem, we want to "decode" the string so that the best individual displays the target forwards.
In order to do this, you'll need to come up with a fitness function that measures how successful candidates are towards this goal.
The constraint is that you may not perform any global operations on the target or individuals.
Your fitness function must still compare a single gene against a single gene.
Your solution will likely not be Pythonic but use indexing.
That's ok.
<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        You may not reverse an entire string (either target or candidate) at any time.
        Everything must be a computation of one gene against one gene (one letter against one letter).
        Failure to follow these directions will result in 0 points for the problem.
    </p>
</div>

The best individual in the population is the one who expresses this string *forwards*.

"this is so much fun"

In [23]:
target2 = "nuf hcum os si siht"

In [24]:
parameters2 = {'population_size': 500, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}

In [25]:
result2 = genetic_algorithm(parameters2, target2, ALPHABET, fitness_evaluation_reverse)

Generation: 0
Best Fitness Score: 4
Best Genotype: ['a', 'k', 'l', 'n', 'f', 'd', 'z', ' ', 'd', 'w', 'e', 'a', 'x', 'c', 'd', ' ', 'f', 'c', 'v']
Best Phenotype: aklnfdz dweaxcd fcv
Generation: 10
Best Fitness Score: 10
Best Genotype: ['t', 'l', 'i', 's', ' ', 'w', 'y', 'u', 'c', 'r', ' ', 'm', 'o', 'c', 'e', ' ', 'f', 'u', 'z']
Best Phenotype: tlis wyucr moce fuz
Generation: 20
Best Fitness Score: 13
Best Genotype: ['t', 'l', 'i', 's', ' ', 'i', 's', 'm', 'c', 'r', ' ', 'm', 'u', 'c', 'c', ' ', 'f', 'u', 'c']
Best Phenotype: tlis ismcr mucc fuc
Generation: 30
Best Fitness Score: 15
Best Genotype: ['t', 'h', 'i', 's', ' ', 'i', 's', 'o', 'r', ' ', ' ', 'm', 'u', 'c', 'e', ' ', 'f', 'u', 'n']
Best Phenotype: this isor  muce fun
Generation: 40
Best Fitness Score: 16
Best Genotype: ['t', 'l', 'i', 's', ' ', 'i', 's', 'b', 's', 'o', ' ', 'm', 'u', 'c', 'h', ' ', 'f', 'u', 'c']
Best Phenotype: tlis isbso much fuc
Generation: 50
Best Fitness Score: 16
Best Genotype: ['t', 'h', 'i', 's', ' '

In [26]:
pprint(result2, compact=True)

{'Best Fitness Score': '19',
 'Best Genotype': "['t', 'h', 'i', 's', ' ', 'i', 's', ' ', 's', 'o', ' ', "
                  "'m', 'u', 'c', 'h', ' ', 'f', 'u', 'n']",
 'Best Phenotype': 'this is so much fun',
 'Generation': 494}


## Problem 3

This is a variation on the theme of Problem 2.
The Caeser Cypher replaces each letter of a string with the letter 13 characters down alphabet (rotating from "z" back to "a" as needed).
This is also known as ROT13 (for "rotate 13").
Latin did not have spaces (and the space is not continguous with the letters a-z) so we'll remove them from our alphabet.
Again, the goal is to derive a fitness function that compares a single gene against a single gene, without global transformations.
This fitness function assigns higher scores to individuals that correctly decode the target.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        You may not apply ROT13 to an entire string (either target or candidate) at any time.
        Everything must be a computation of one gene against one gene.
        Failure to follow these directions will result in 0 points for the problem.
    </p>
</div>

The best individual will express the target *decoded*.

"thisissomuchfun"

In [27]:
ALPHABET3 = "abcdefghijklmnopqrstuvwxyz"

In [28]:
target3 = "guvfvffbzhpusha"

In [29]:
parameters3 = {'population_size': 500, 'crossover_rate': 0.8, 'mutation_rate': 0.2, 'generation_count': 100000}

In [30]:
result3 = genetic_algorithm(parameters3, target3, ALPHABET3, fitness_evaluation_rot13) # do what you need to do for your implementation but don't change the lines above or below.

Generation: 0
Best Fitness Score: 4
Best Genotype: ['t', 'x', 'i', 'h', 's', 'v', 'h', 't', 'e', 'c', 'c', 'c', 'y', 'u', 'q']
Best Phenotype: txihsvhtecccyuq
Generation: 10
Best Fitness Score: 9
Best Genotype: ['t', 'x', 'i', 's', 'a', 's', 'x', 'o', 'm', 'c', 'l', 'h', 'f', 'l', 'n']
Best Phenotype: txisasxomclhfln
Generation: 20
Best Fitness Score: 11
Best Genotype: ['t', 'x', 'i', 'y', 's', 's', 'y', 'o', 'm', 'u', 'c', 'h', 'f', 'u', 'n']
Best Phenotype: txiyssyomuchfun
Generation: 30
Best Fitness Score: 13
Best Genotype: ['t', 'h', 'i', 's', 'a', 's', 's', 'o', 'm', 'u', 'c', 'h', 'e', 'u', 'n']
Best Phenotype: thisassomucheun
Generation: 40
Best Fitness Score: 13
Best Genotype: ['t', 'h', 's', 's', 'z', 's', 's', 'o', 'm', 'u', 'c', 'h', 'f', 'u', 'n']
Best Phenotype: thsszssomuchfun
Generation: 50
Best Fitness Score: 13
Best Genotype: ['t', 'h', 'i', 's', 'z', 's', 's', 'o', 'm', 'u', 'c', 'a', 'f', 'u', 'n']
Best Phenotype: thiszssomucafun
Generation: 60
Best Fitness Score: 14

In [31]:
pprint(result3, compact=True)

{'Best Fitness Score': '15',
 'Best Genotype': "['t', 'h', 'i', 's', 'i', 's', 's', 'o', 'm', 'u', 'c', "
                  "'h', 'f', 'u', 'n']",
 'Best Phenotype': 'thisissomuchfun',
 'Generation': 89}


## Problem 4

## In Problem 3, we assumed we knew what the shift was in ROT-13.
## What if we didn't?
## Describe how you might solve that problem including a description of the solution encoding (chromosome and interpretation) and fitness function. Assume we can add spaces into the message.

When using the 26-letter alphabet, there are a maximum of 25 different shifts that can be done before the shifts begin to loop back to the original string. Each time the fitness function is evaluated, it may be possible to generate a score for each of the 25 shifts after crossover and mutation. This can be done to simultaneously find the aprropriate shift, but also evaluate the best fitness of the child. 

For example: if the child is ['a', 'b', 'c'], the following would be evaluated against the target string for a fitness score: 
['a', 'b', 'c'],
['b', 'c', 'd'],
['c', 'd', 'e'],
['d', 'e', 'f'],
... 
['z', 'a', 'b']

## Challenge

**You do not need to do this problem and it won't be graded if you do. It's just here if you want to push your understanding.**

The original GA used binary encodings for everything.
We're basically using a Base 27 encoding.
You could, however, write a version of the algorithm that uses an 8 bit encoding for each letter (ignore spaces as they're a bit of a bother).
That is, a 4 letter candidate looks like this:

```
0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1
```

If you wrote your `genetic_algorithm` code general enough, with higher order functions, you should be able to implement it using bit strings instead of latin strings.

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.